# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer dataset using the `mlcroissant` library, following modern best practices for referencing schema entities by their `@id` and dynamic, reproducible data exploration.

### Dataset Source

The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and all records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata (will download and parse the Croissant schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their unique `@id`s.

Let's enumerate the available record sets, and for each, list their fields (and corresponding `@id`s).

In [ ]:
# List all record sets and their field @id's
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    # List fields for this record set
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  Field: {field['@id']} (name: {field.get('name', '')})")
        else:
            print(f"  Field: {field}")

For quick exploration, print out the first 2 records from each record set.

In [ ]:
for record_set in record_sets:
    rs_id = record_set['@id']
    print(f"\nSample records from record set '@id': {rs_id}")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if idx >= 1:
                break
    except Exception as e:
        print(f"Could not load records for {rs_id}. Error: {e}")

## 3. Data Extraction

Load all available record sets into pandas DataFrames for further analysis. Each DataFrame key is the `@id` of the record set, and fields/columns maintain their original Croissant `@id` names where possible.

In [ ]:
dataframes = {}

# Collect all record set @ids for convenience
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"There are {len(record_set_ids)} record sets:")
print(record_set_ids)

# Load all record sets into pandas DataFrames
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

# Show fields/columns in the main clinical data record set if available
# (User should set the correct primary record set @id for analysis)
main_record_set_id = None
for r in record_set_ids:
    if 'clinical' in r.lower() or 'colorectal' in r.lower():
        main_record_set_id = r
        break
if not main_record_set_id:
    main_record_set_id = record_set_ids[0]  # Default to first if unsure
print(f"\nColumns in primary record set '@id': {main_record_set_id}")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll now explore the data:
- Select a relevant numeric field (such as 'age', 'interval', or biometric measurement) referenced by its `@id`.
- Filter records based on a threshold, normalize, and group by a categorical field, with all references by `@id`.

In [ ]:
# Replace the field @id's below to match those in your record set
# Use the .columns output above to help select fields.

df = dataframes[main_record_set_id]  # Alias for easier access

# Example: suppose the numeric age field is '@id': 'http://mlcommons.org/croissant/field/age'
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if ('sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower() or 'status' in col.lower()) and not group_field_id:
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
print(f"Group by field: {group_field_id}")

# Convert numeric field to numeric in case it's string-typed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = 60
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or field relationships. The following cell shows histograms and, if grouping is possible, boxplots by the chosen categorical field. All axes use Croissant field `@id`s for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

This notebook demonstrates how to use `mlcroissant` to programmatically load and explore datasets described by detailed Croissant schemas. The FAIR² colorectal cancer dataset illustrates:
- Access to clinical and pathological variables via unique `@id` references
- Filtering and transforming numeric fields for cohort analysis
- Quick grouping and visualization of key metrics using robust identifiers.

Extend this template for your data science and clinical research by identifying the relevant `@id`s for your use case from the Croissant schema documentation or by inspecting the fields above.
